<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-07-fine-tuning/lesson-7.4-open-source-fine-tuning/notebooks/exercises-7_4.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 7.4: Open-Source Fine-Tuning

*Module 7 · Fine-Tuning LLMs LIVE CLASS*

> Closed APIs (OpenAI, Vertex AI) are easy but give you no control. Open-source fine-tuning gives you EVERYTHING: custom loss functions, training loops, quantization, multi-GPU, and 2x speed with Unsloth. This lesson codes the full pipeline: load model, prepare data, configure LoRA, train, evaluate, and push to Hub.

`HF Trainer` · `Unsloth 2x` · `LoRA/QLoRA` · `Vertex AI GPU` · `60 min`

---

## About this notebook

This notebook contains the **3 runnable code examples** from the published lesson `lesson-7.4.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — HuggingFace SFTTrainer — The Standard Pipeline — `01_hf_trainer.py`
2. Step 2 — Unsloth — 2x Faster, 60% Less Memory — `02_unsloth.py`
3. Step 3 — Vertex AI Training — Managed GPU at Scale — `03_vertex_training.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q transformers torch datasets peft bitsandbytes trl accelerate


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · HuggingFace SFTTrainer — The Standard Pipeline

Load model + LoRA + data + train. The industry-standard approach.

**`01_hf_trainer.py`** · _Block 1 of 3_

HUGGINGFACE SFT TRAINER — Standard fine-tuning pipeline — pip install transformers datasets peft trl bitsandbytes accelerate


In [ ]:
# HUGGINGFACE SFT TRAINER — Standard fine-tuning pipeline
# pip install transformers datasets peft trl bitsandbytes accelerate
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
import torch

# ── 1. Load model with 4-bit quantization (QLoRA) ──
model_name = "google/gemma-2-2b-it"  # or "meta-llama/Llama-3.1-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name, quantization_config=bnb_config, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# ── 2. Configure LoRA ──
lora_config = LoraConfig(
    r=16,                    # rank (lower = fewer params)
    lora_alpha=32,            # scaling factor
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({trainable/total*100:.2f}%)")

# ── 3. Load dataset ──
# dataset = load_dataset("json", data_files="netsetos_train.jsonl")
# Or use a HuggingFace dataset:
# dataset = load_dataset("your-org/netsetos-support", split="train")

# ── 4. Training config ──
training_config = SFTConfig(
    output_dir="./netsetos-gemma-lora",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,  # effective batch = 16
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=10,
    save_strategy="epoch",
    bf16=True,
    max_seq_length=2048,
)

# ── 5. Train ──
# trainer = SFTTrainer(
#     model=model, args=training_config,
#     train_dataset=dataset["train"],
#     tokenizer=tokenizer,
# )
# trainer.train()
# model.save_pretrained("./netsetos-gemma-lora")

print("\nHuggingFace SFT Pipeline:")
print("  1. Load model + 4-bit quantization (QLoRA)")
print("  2. Add LoRA adapters (0.5-2% trainable params)")
print("  3. Load JSONL dataset")
print("  4. Configure: 3 epochs, lr=2e-4, cosine schedule")
print("  5. Train + save LoRA weights (~50MB)")


> **What just happened?** A 2B-parameter model loaded in 4-bit quantization (QLoRA). LoRA adapters added to attention layers — only 0.5-2% of parameters are trainable. SFTTrainer handles the training loop: batching, gradient accumulation, learning rate scheduling, checkpointing. Result: a 50MB LoRA adapter file that transforms a general model into YOUR domain-specific model. Full model stays frozen. Only the adapter is trained and saved.


### Step 2 · Unsloth — 2x Faster, 60% Less Memory

Drop-in replacement for HF Trainer. Custom CUDA kernels make it dramatically faster.

**`02_unsloth.py`** · _Block 2 of 3_

UNSLOTH — 2x faster fine-tuning — pip install unsloth


In [ ]:
# UNSLOTH — 2x faster fine-tuning
# pip install unsloth
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig

# ── 1. Load model (Unsloth handles quantization automatically) ──
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gemma-2-2b-it-bnb-4bit",  # pre-quantized
    max_seq_length=2048,
    load_in_4bit=True,
)

# ── 2. Add LoRA (Unsloth-optimized) ──
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0,        # Unsloth: 0 dropout = faster
    use_gradient_checkpointing="unsloth",  # 60% less VRAM
)

# ── 3. Chat template for data formatting ──
def format_chat(examples):
    texts = []
    for msgs in examples["messages"]:
        text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return {"text": texts}

# ── 4. Train (same SFTTrainer, but 2x faster!) ──
# trainer = SFTTrainer(
#     model=model, tokenizer=tokenizer,
#     train_dataset=dataset.map(format_chat, batched=True),
#     dataset_text_field="text",
#     args=SFTConfig(output_dir="./netsetos-unsloth",
#                    num_train_epochs=3, per_device_train_batch_size=4,
#                    learning_rate=2e-4, bf16=True, max_seq_length=2048),
# )
# trainer.train()

# ── 5. Save & export ──
# model.save_pretrained_merged("netsetos-merged", tokenizer)
# model.save_pretrained_gguf("netsetos.gguf", tokenizer)  # for llama.cpp

print("Unsloth Pipeline:")
print("  Same code as HF Trainer, but:")
print("  - 2x faster training (custom CUDA kernels)")
print("  - 60% less VRAM (gradient checkpointing)")
print("  - Export to GGUF (for llama.cpp / Ollama)")
print("  - Free tier works on Colab T4 GPU")


> **What just happened?** Same pipeline as HF Trainer, but FastLanguageModel replaces standard loading. Custom CUDA kernels make training 2x faster. use_gradient_checkpointing="unsloth" saves 60% VRAM. You can export directly to GGUF for running on llama.cpp or Ollama. Unsloth on a free Colab T4 can fine-tune a 7B model — something that normally needs an A100.


### Step 3 · Vertex AI Training — Managed GPU at Scale

When Colab isn’t enough: A100/H100 on GCP with managed training jobs.

**`03_vertex_training.py`** · _Block 3 of 3_

VERTEX AI CUSTOM TRAINING JOB — Run HF Trainer / Unsloth on managed A100 GPUs


In [ ]:
# VERTEX AI CUSTOM TRAINING JOB
# Run HF Trainer / Unsloth on managed A100 GPUs
from google.cloud import aiplatform

aiplatform.init(project="netsetos-ai", location="us-central1")

# ── Option 1: Custom container job ──
job = aiplatform.CustomContainerTrainingJob(
    display_name="netsetos-gemma-finetune",
    container_uri="us-docker.pkg.dev/netsetos-ai/training/finetune:latest",
    model_serving_container_image_uri="us-docker.pkg.dev/vertex-ai/prediction/pytorch-gpu:latest",
)

# ── Run on A100 ──
# model = job.run(
#     replica_count=1,
#     machine_type="a2-highgpu-1g",     # 1x A100 40GB
#     accelerator_type="NVIDIA_TESLA_A100",
#     accelerator_count=1,
#     args=["--model", "google/gemma-2-2b-it",
#           "--data", "gs://netsetos-data/train.jsonl",
#           "--epochs", "3", "--lr", "2e-4"],
# )

print("Vertex AI Training Options:\n")
gpus = [
    ("T4",    "16GB",  "~$0.35/hr",  "2B models, QLoRA"),
    ("L4",    "24GB",  "~$0.70/hr",  "7B models, QLoRA"),
    ("A100",  "40GB",  "~$3.67/hr",  "13B models, full LoRA"),
    ("A100",  "80GB",  "~$5.12/hr",  "70B models, QLoRA"),
    ("H100",  "80GB",  "~$8.00/hr",  "70B+ models, fastest"),
]
print(f"  {'GPU':7} {'VRAM':7} {'Cost':11} Use Case")
for g,v,c,u in gpus:
    print(f"  {g:7} {v:7} {c:11} {u}")
print(f"\n  Netsetos recommendation: L4 for dev, A100-40GB for production")


> **What just happened?** The export pipeline: Train → Merge → Quantize → Serve . Merge eliminates LoRA overhead at inference time. GGUF Q4_K_M compresses to ~25% of original size with good quality retention. AWQ is the GPU standard — INT4 quantization with activation-aware weight protection. Unsloth handles the entire pipeline in a single API call: save_pretrained_gguf().


---

## Wrap-up

You've walked through all 3 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
